# 🔥 OTAKU UPSCALER V2 — TOPAZ INSPIRED
**Image + Video • Anime/Movie • Real-ESRGAN + Anime 6B**

In [ ]:
# GPU + FFMPEG CHECK
import torch
!ffmpeg -version | head -n 1
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ENVIRONMENT
!apt-get -qq update && apt-get -qq install -y python3.11 python3.11-venv ffmpeg
!python3.11 -m venv /content/venv311
!/content/venv311/bin/python -m pip install -q --upgrade pip
!/content/venv311/bin/pip install -q basicsr facexlib gfpgan gradio Pillow opencv-python numpy
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /content/venv311/lib/python3.11/site-packages/basicsr/data/degradations.py
print("✅ V2 ENV READY")


In [ ]:
# REAL-ESRGAN ENGINES + 2 MODELS
%cd /content
!rm -rf /content/realesrgan
!git clone -q https://github.com/xinntao/Real-ESRGAN.git /content/realesrgan
!echo '__version__ = "0.3.0"' > /content/realesrgan/realesrgan/version.py
!mkdir -p /content/realesrgan/weights
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -O /content/realesrgan/weights/RealESRGAN_x4plus.pth
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth -O /content/realesrgan/weights/RealESRGAN_x4plus_anime_6B.pth
print("✅ RealESRGAN_x4plus + anime_6B ready")


In [ ]:
# 🔥 OTAKU V2 — IMAGE + VIDEO UI
import os, subprocess, uuid, shutil, cv2
from pathlib import Path
import torch
import gradio as gr
from PIL import Image, ImageFilter

REAL_ESRGAN = Path("/content/realesrgan")
VENV_PYTHON = "/content/venv311/bin/python"
WORK_DIR = Path("/content/otaku_v2")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
FRAMES_DIR = WORK_DIR / "frames"
UPSCALED_DIR = WORK_DIR / "frames_upscaled"
for d in [INPUT_DIR, OUTPUT_DIR, FRAMES_DIR, UPSCALED_DIR]: d.mkdir(parents=True, exist_ok=True)

# Your Topaz reference - now APPLIED
OTAKU_TOPAZ_REFERENCE = {
 "before_cc": {"revert_compression":74,"recover_details":88,"sharpen":40,"reduce_noise":59,"dehalo":13,"antialias_deblur":-22},
 "after_cc": {"revert_compression":65,"recover_details":86,"sharpen":10,"reduce_noise":62,"dehalo":15,"antialias_deblur":-49}
}

def apply_topaz_cc(pil_img, cc_profile):
 # Simple Topaz-inspired simulation: sharpen + denoise
 sharpen_val = cc_profile["sharpen"] / 100.0
 if sharpen_val > 0.1:
 pil_img = pil_img.filter(ImageFilter.UnsharpMask(radius=2, percent=int(sharpen_val*300), threshold=2))
 if cc_profile["reduce_noise"] > 50:
 pil_img = pil_img.filter(ImageFilter.MedianFilter(size=3))
 return pil_img

def run_realesrgan(input_path, output_dir, scale, model_name):
 cmd = [VENV_PYTHON, "inference_realesrgan.py", "-n", model_name, "-i", str(input_path), "-o", str(output_dir), "--outscale", str(scale), "--tile", "256", "--tile_pad", "10", "--fp32"]
 proc = subprocess.run(cmd, cwd=str(REAL_ESRGAN), text=True, capture_output=True)
 if proc.returncode!= 0:
 print(proc.stderr); raise gr.Error("Real-ESRGAN failed. Check logs.")
 return proc

def otaku_upscale_image(image, scale, mode):
 if image is None: raise gr.Error("Upload image first")
 job_id = uuid.uuid4().hex[:6]
 in_path = INPUT_DIR / f"img_{job_id}.png"
 # BEFORE CC
 image = apply_topaz_cc(image.convert("RGB"), OTAKU_TOPAZ_REFERENCE["before_cc"])
 image.save(in_path)
 model = "RealESRGAN_x4plus_anime_6B" if mode=="Anime" else "RealESRGAN_x4plus"
 scale_v = int(scale.replace("×",""))
 for f in OUTPUT_DIR.iterdir(): f.unlink()
 run_realesrgan(in_path, OUTPUT_DIR, scale_v, model)
 out_file = list(OUTPUT_DIR.glob("*_out.*"))[0]
 result = Image.open(out_file).convert("RGB")
 # AFTER CC
 result = apply_topaz_cc(result, OTAKU_TOPAZ_REFERENCE["after_cc"])
 final_path = OUTPUT_DIR / f"otaku_final_{job_id}.png"
 result.save(final_path)
 info = f"🔥 V2 COMPLETE\nMode: {mode}\nModel: {model}\nScale: {scale_v}x\nOriginal: {Image.open(in_path).size} -> {result.size}\nGPU: {torch.cuda.get_device_name(0)}"
 return result, info, str(final_path)

def otaku_upscale_video(video_path, scale, mode, out_format):
 if video_path is None: raise gr.Error("Upload video first")
 job_id = uuid.uuid4().hex[:6]
 scale_v = int(scale.replace("×",""))
 model = "RealESRGAN_x4plus_anime_6B" if mode=="Anime" else "RealESRGAN_x4plus"
 # Clean
 for d in [FRAMES_DIR, UPSCALED_DIR]: 
 for f in d.iterdir(): f.unlink() if f.is_file() else shutil.rmtree(f)
 for f in OUTPUT_DIR.iterdir(): f.unlink()
 # 1. Extract frames
 cap = cv2.VideoCapture(video_path)
 fps = cap.get(cv2.CAP_PROP_FPS) or 24
 idx=0
 while True:
 ret, frame = cap.read()
 if not ret: break
 cv2.imwrite(str(FRAMES_DIR / f"{idx:06d}.png"), frame)
 idx+=1
 cap.release()
 print(f"Extracted {idx} frames at {fps} fps")
 # 2. Upscale frames folder
 run_realesrgan(FRAMES_DIR, UPSCALED_DIR, scale_v, model)
 # 3. Re-encode
 final_video = OUTPUT_DIR / f"otaku_video_{job_id}.{'mov' if 'ProRes' in out_format else 'mp4'}"
 if 'ProRes' in out_format:
 ffmpeg_cmd = ["ffmpeg","-y","-framerate",str(fps),"-i",str(UPSCALED_DIR / "%06d_out.png"),"-c:v","prores_ks","-profile:v","3","-pix_fmt","yuv422p10le",str(final_video)]
 else:
 ffmpeg_cmd = ["ffmpeg","-y","-framerate",str(fps),"-i",str(UPSCALED_DIR / "%06d_out.png"),"-c:v","libx264","-pix_fmt","yuv420p","-crf","16",str(final_video)]
 subprocess.run(ffmpeg_cmd, check=True)
 info = f"🔥 VIDEO DONE\nFrames: {idx}\nFPS: {fps}\nScale: {scale_v}x\nModel: {model}\nFormat: {out_format}"
 return str(final_video), info, str(final_video)

with gr.Blocks(title="OTAKU V2") as app:
 gr.Markdown("# 🔥 OTAKU UPSCALER V2\n### Image + Video • Anime + Movie • Topaz Proteus Inspired")
 with gr.Tabs():
 with gr.Tab("📸 Image"):
 with gr.Row():
 with gr.Column():
 img_in = gr.Image(type="pil", label="📥 Upload Image")
 img_scale = gr.Radio(["2×","4×"], value="2×", label="Scale")
 img_mode = gr.Radio(["Anime","Movie"], value="Anime", label="Mode")
 btn_img = gr.Button("🔥 UPSCALE IMAGE", variant="primary")
 with gr.Column():
 img_out = gr.Image(type="pil", label="✨ Result")
 img_info = gr.Textbox(lines=7, label="Info")
 img_dl = gr.File(label="Download")
 with gr.Tab("🎬 Video"):
 with gr.Row():
 with gr.Column():
 vid_in = gr.Video(label="📥 Upload Video")
 vid_scale = gr.Radio(["2×","4×"], value="2×", label="Scale")
 vid_mode = gr.Radio(["Anime","Movie"], value="Anime", label="Mode")
 vid_fmt = gr.Radio(["MP4 - H264","MOV - ProRes 422 HQ"], value="MP4 - H264", label="Output Format (Your Topaz ref: MOV ProRes)")
 btn_vid = gr.Button("🔥 UPSCALE VIDEO", variant="primary")
 with gr.Column():
 vid_out = gr.Video(label="✨ Result")
 vid_info = gr.Textbox(lines=7, label="Info")
 vid_dl = gr.File(label="Download")
 btn_img.click(fn=otaku_upscale_image, inputs=[img_in, img_scale, img_mode], outputs=[img_out, img_info, img_dl])
 btn_vid.click(fn=otaku_upscale_video, inputs=[vid_in, vid_scale, vid_mode, vid_fmt], outputs=[vid_out, vid_info, vid_dl])
 gr.Markdown("---\n**OTAKU V2**: Before CC + After CC now active. Anime 6B = best for your JJK edits.")
app.launch(share=True, debug=True)
